# Breast Cancer Classification using K-Nearest Neighbors (KNN)

**Dataset:** [Breast Cancer Wisconsin Diagnostic Dataset (Kaggle)](https://www.kaggle.com/datasets/uciml/breast-cancer-wisconsin-data)

**Goal:** Build a K-Nearest Neighbors (KNN) classification model to predict whether a breast tumor
is **Malignant (M)** or **Benign (B)** based on diagnostic measurements computed from digitized
images of a fine needle aspirate (FNA) of a breast mass.


## Setup

Import the libraries used throughout the notebook.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)

sns.set(style="whitegrid")
%matplotlib inline


## Task 1: Data Understanding (2 Marks)

1. Load the dataset using Pandas.
2. Display the first five records.
3. Identify numerical features and the target variable.
4. Display dataset information and summary statistics.

> **Note:** Download `data.csv` (the Breast Cancer Wisconsin Diagnostic dataset) from the Kaggle
> link above and place it in the same folder as this notebook before running the cell below.


In [ ]:
# 1. Load the dataset
df = pd.read_csv('data.csv')

print("Shape of dataset:", df.shape)
df.head()


In [ ]:
# Dataset information
df.info()


In [ ]:
# Summary statistics
df.describe()


### Feature identification

The Breast Cancer Wisconsin Diagnostic dataset typically contains:

- **An identifier column:** `id` — not predictive, will be dropped.
- **An unnamed/empty trailing column** (`Unnamed: 32`) that sometimes appears due to a trailing
  comma in the raw CSV — will be dropped if present.
- **Numerical features (30 in total):** computed measurements of cell nuclei from the digitized
  image, in three groups of 10 — `mean`, `standard error (se)`, and `worst` values — for
  attributes like `radius`, `texture`, `perimeter`, `area`, `smoothness`, `compactness`,
  `concavity`, `concave points`, `symmetry`, and `fractal_dimension`.
- **Target variable:** `diagnosis` — `M` (Malignant) or `B` (Benign).


In [ ]:
target_variable = 'diagnosis'
numerical_features = [col for col in df.columns if col not in ['id', 'diagnosis', 'Unnamed: 32']]

print("Number of numerical features:", len(numerical_features))
print("Numerical features:", numerical_features)
print("Target variable:", target_variable)

# Class distribution
print("\nDiagnosis class distribution:")
print(df['diagnosis'].value_counts())

sns.countplot(x='diagnosis', data=df)
plt.title('Diagnosis Class Distribution (M = Malignant, B = Benign)')
plt.show()


## Task 2: Data Preprocessing (2 Marks)

- Check for missing values.
- Remove unnecessary columns (if any).
- Encode the target variable if required.
- Normalize or standardize the feature values.
- Split the dataset into 80% training and 20% testing.


In [ ]:
# Check for missing values
print("Missing values per column (top 10):")
print(df.isnull().sum().sort_values(ascending=False).head(10))


In [ ]:
df_clean = df.copy()

# Remove unnecessary columns: 'id' (identifier) and 'Unnamed: 32' (empty trailing column, if present)
cols_to_drop = [col for col in ['id', 'Unnamed: 32'] if col in df_clean.columns]
df_clean.drop(columns=cols_to_drop, inplace=True)

print("Dropped columns:", cols_to_drop)
print("Missing values remaining after cleanup:", df_clean.isnull().sum().sum())


In [ ]:
# Encode the target variable: M -> 1 (Malignant), B -> 0 (Benign)
le = LabelEncoder()
df_clean['diagnosis'] = le.fit_transform(df_clean['diagnosis'])  # B=0, M=1

print("Encoded classes:", dict(zip(le.classes_, le.transform(le.classes_))))
df_clean['diagnosis'].value_counts()


In [ ]:
# Features and target
X = df_clean.drop('diagnosis', axis=1)
y = df_clean['diagnosis']

# Split into 80% training and 20% testing (stratified to preserve class ratio)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# Standardize feature values - CRITICAL for KNN, since it relies on distance calculations
# and features on larger numeric scales (e.g. 'area') would otherwise dominate the distance.
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Training set size:", X_train_scaled.shape)
print("Testing set size :", X_test_scaled.shape)


## Task 3: Model Development (3 Marks)

1. Train a K-Nearest Neighbors (KNN) classifier.
2. Use K = 5 as the initial value.
3. Predict the class labels for the test dataset.


In [ ]:
# 1 & 2. Train the KNN classifier with K = 5
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train_scaled, y_train)

# 3. Predict class labels for the test dataset
y_pred = knn.predict(X_test_scaled)

print("Predicted labels for first 10 test samples:", y_pred[:10])
print("Actual labels for first 10 test samples   :", y_test.values[:10])


## Task 4: Model Evaluation (2 Marks)

Evaluate the model using Accuracy, Precision, Recall, and F1-Score, and generate a confusion
matrix.


In [ ]:
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print(f"Accuracy Score: {accuracy:.4f}")
print(f"Precision     : {precision:.4f}")
print(f"Recall        : {recall:.4f}")
print(f"F1-Score      : {f1:.4f}")

print("\nFull classification report:")
print(classification_report(y_test, y_pred, target_names=['Benign', 'Malignant']))


In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Benign', 'Malignant'],
            yticklabels=['Benign', 'Malignant'])
plt.xlabel('Predicted Label')
plt.ylabel('Actual Label')
plt.title('Confusion Matrix - Breast Cancer Classification (KNN, K=5)')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150)
plt.show()


In [ ]:
# Bonus: check how accuracy varies with different values of K (helps justify K=5 choice)
k_values = range(1, 21)
accuracies = []

for k in k_values:
    knn_k = KNeighborsClassifier(n_neighbors=k)
    knn_k.fit(X_train_scaled, y_train)
    acc_k = accuracy_score(y_test, knn_k.predict(X_test_scaled))
    accuracies.append(acc_k)

plt.figure(figsize=(8, 5))
plt.plot(k_values, accuracies, marker='o', color='navy')
plt.axvline(x=5, color='red', linestyle='--', label='K = 5 (assignment value)')
plt.xlabel('Number of Neighbors (K)')
plt.ylabel('Test Accuracy')
plt.title('KNN Accuracy vs K')
plt.xticks(list(k_values))
plt.legend()
plt.tight_layout()
plt.show()


### Observations

1. With **K = 5** and standardized features, the KNN classifier achieves high accuracy, precision,
   recall, and F1-Score on the test set, showing that tumor diagnosis measurements (radius, texture,
   concavity, etc.) separate malignant and benign cases fairly well in feature space.
2. The confusion matrix shows the model makes very few misclassifications overall; in a medical
   context, **false negatives** (malignant tumors predicted as benign) are the more dangerous error
   type, so it's worth checking that cell of the confusion matrix specifically rather than relying
   on accuracy alone.
3. The K-vs-accuracy plot shows accuracy is relatively stable across a range of K values, but very
   small K (like K=1) can be more sensitive to noisy/mislabeled points, while very large K can start
   to blur the boundary between classes — K=5 is a reasonable middle ground, consistent with the
   assignment's chosen value.

*(Exact numeric values for Accuracy, Precision, Recall, and F1-Score will depend on the actual
`data.csv` split used when you run this notebook — update these observations based on your own
printed results.)*


## Task 5: Conclusion (1 Mark)

This project used K-Nearest Neighbors (KNN) to classify breast tumors as Malignant or Benign based
on 30 diagnostic measurements derived from digitized images of fine needle aspirates. After
removing unnecessary columns (`id` and the empty trailing column), encoding the target variable,
standardizing all feature values, and splitting the data 80/20, a KNN classifier with K = 5 was
trained and evaluated using Accuracy, Precision, Recall, F1-Score, and a Confusion Matrix.

The key finding is that tumor measurements like radius, concavity, and concave points are strong
discriminators between malignant and benign tumors, allowing KNN to achieve strong classification
performance even with a relatively simple, non-parametric algorithm.

**Feature scaling is critical for KNN** because the algorithm classifies a new point based on the
distance to its nearest neighbors; if features are left on their original scales (e.g., `area` in
the hundreds vs `smoothness` as a small decimal), features with larger numeric ranges would
dominate the distance calculation and distort the results, regardless of their actual importance.

A key **limitation of KNN** is that it is computationally expensive at prediction time, since it
must compute the distance from a new point to every point in the training set — this doesn't scale
well to very large datasets, and KNN also has no explicit "model" to interpret afterward the way
Logistic Regression's coefficients can be interpreted.
